# 🔌 第6章：MCP 协议与扩展系统

## 🎯 学习目标
1. 理解 MCP (Model Context Protocol) 的核心概念
2. 理解 Claude Code 的 MCP 客户端实现
3. 理解 MCP 工具如何被包装为内置工具
4. 理解 Skills 和 Plugins 扩展系统

---

## 6.1 MCP 是什么？

**Model Context Protocol (MCP)** 是 Anthropic 提出的开放协议，
用于标准化 AI 模型与外部工具/数据源的交互。

```
┌──────────────┐     MCP Protocol      ┌──────────────┐
│  Claude Code  │ ◄──────────────────► │  MCP Server   │
│  (MCP Client) │     JSON-RPC 2.0     │  (任何语言)    │
└──────────────┘                       └──────────────┘
                                        可以提供：
                                        - Tools (工具)
                                        - Resources (资源)
                                        - Prompts (提示词)
```

### MCP 的三大能力

| 能力 | 说明 | 示例 |
|------|------|------|
| **Tools** | 可执行的操作 | 数据库查询、API 调用、文件操作 |
| **Resources** | 可读取的数据 | 文档、配置、实时数据 |
| **Prompts** | 预定义的提示词模板 | 代码审查模板、分析模板 |

### 传输方式

```typescript
// Claude Code 支持多种 MCP 传输方式
type McpTransport = 
  | StdioClientTransport      // 标准输入/输出（最常用）
  | SSEClientTransport        // Server-Sent Events
  | StreamableHTTPClientTransport  // HTTP 流
  | WebSocketTransport        // WebSocket
  | SdkControlClientTransport // SDK 控制通道
```

## 6.2 MCP 客户端实现 (client.ts)

`services/mcp/client.ts` (116KB, 3349行) 是 MCP 客户端的核心实现。

### 连接管理

```typescript
// 连接到 MCP 服务器
export async function connectToServer(
  name: string, 
  config: ScopedMcpServerConfig
): Promise<MCPServerConnection> {
  // 1. 根据配置创建传输层
  let transport: Transport;
  if (config.command) {
    // Stdio 模式：启动子进程
    transport = new StdioClientTransport({
      command: config.command,
      args: config.args,
      env: { ...subprocessEnv(), ...config.env },
    });
  } else if (config.url) {
    // HTTP/SSE/WebSocket 模式
    transport = createRemoteTransport(config.url, config);
  }
  
  // 2. 创建 MCP Client
  const client = new Client({
    name: 'claude-code',
    version: '1.0.0',
  });
  
  // 3. 连接
  await client.connect(transport);
  
  // 4. 返回连接对象
  return {
    type: 'connected',
    name,
    client,
    cleanup: () => client.close(),
  };
}
```

### 错误处理

```typescript
// 自定义错误类型
class McpAuthError extends Error {}
  // OAuth token 过期 → 需要重新认证

class McpSessionExpiredError extends Error {}
  // 会话过期 → 重新连接

class McpToolCallError extends Error {}
  // 工具调用失败 → 返回错误给 Claude

// 会话过期检测
function isMcpSessionExpiredError(error: Error): boolean {
  // HTTP 404 + JSON-RPC code -32001
  // 表示服务器不再认识这个会话
}
```

## 6.3 MCP 工具包装

MCP 服务器提供的工具会被包装为 Claude Code 的内置 Tool 格式：

```typescript
// MCPTool.ts — MCP 工具包装器
function createMcpTool(serverName, toolDef): Tool {
  return buildTool({
    // 名称格式: mcp__serverName__toolName
    name: `mcp__${normalizeNameForMCP(serverName)}__${normalizeNameForMCP(toolDef.name)}`,
    
    // 使用 MCP 工具的 JSON Schema（不是 Zod）
    inputJSONSchema: toolDef.inputSchema,
    
    isMcp: true,  // 标记为 MCP 工具
    
    // MCP 信息（原始服务器名和工具名）
    mcpInfo: { serverName, toolName: toolDef.name },
    
    // 执行：调用 MCP 服务器
    async call(input, context) {
      const result = await client.callTool({
        name: toolDef.name,
        arguments: input,
      });
      
      // 处理结果
      return processToolResult(result);
    },
    
    // 权限：默认需要用户确认
    async checkPermissions() {
      return {
        behavior: 'passthrough',
        message: 'MCPTool requires permission.',
        suggestions: [{
          type: 'addRules',
          rules: [{ toolName: fullyQualifiedName }],
          behavior: 'allow',
          destination: 'localSettings',
        }],
      };
    },
  });
}
```

### MCP 工具结果处理

```typescript
// MCP 工具可以返回多种内容类型
function processToolResult(result) {
  for (const content of result.content) {
    switch (content.type) {
      case 'text':     // 文本结果
      case 'image':    // 图片（base64）
      case 'resource': // 资源链接
    }
  }
  
  // 大结果处理
  if (mcpContentNeedsTruncation(result)) {
    return truncateMcpContentIfNeeded(result);
  }
  
  // 二进制内容持久化
  if (isBinaryContent(result)) {
    const path = await persistBinaryContent(result);
    return getBinaryBlobSavedMessage(path);
  }
}
```

## 6.4 MCP 配置

MCP 服务器可以在多个层级配置：

```json
// ~/.claude/settings.json (用户级)
{
  "mcpServers": {
    "my-database": {
      "command": "node",
      "args": ["db-server.js"],
      "env": { "DB_URL": "postgres://..." }
    },
    "web-api": {
      "url": "https://api.example.com/mcp",
      "transport": "sse"
    }
  }
}

// .claude/settings.json (项目级)
{
  "mcpServers": {
    "project-tools": {
      "command": "python",
      "args": ["tools/mcp_server.py"]
    }
  }
}
```

### 配置合并逻辑

```typescript
// config.ts
function getAllMcpConfigs(): Map<string, ScopedMcpServerConfig> {
  // 1. 用户级配置
  // 2. 项目级配置
  // 3. 企业策略配置
  // 4. Claude.ai 配置（如果是 Claude.ai 订阅者）
  // 项目级覆盖用户级（同名时）
}
```

## 6.5 Skills 系统

Skills 是可复用的工作流，本质上是预定义的 prompt 模板：

```
skills/
├── bundled/              # 内置技能
│   ├── index.ts          # 注册入口
│   └── ...               # 各种内置技能
└── bundledSkills.ts      # 技能加载器
```

### 技能定义

```markdown
<!-- .claude/skills/review-pr.md -->
---
name: review-pr
description: Review a pull request
---

# PR Review Skill

Please review the current PR changes:

1. Run `git diff main...HEAD` to see changes
2. Check for:
   - Code quality issues
   - Security vulnerabilities
   - Missing tests
3. Provide a summary with recommendations
```

### 技能调用

```typescript
// 通过 SkillTool 调用
SkillTool.call({ skill: 'review-pr', args: '' });

// 或通过 /slash 命令
// 用户输入: /review-pr
```

## 6.6 Plugins 系统

Plugins 是更强大的扩展机制，可以提供工具、技能、Agent 等：

```
plugins/
├── bundled/              # 内置插件
│   └── index.ts          # 注册入口
└── builtinPlugins.ts     # 插件加载器
```

### 插件能力

```typescript
type Plugin = {
  name: string;
  version: string;
  
  // 可以提供的能力
  tools?: Tool[];           // 自定义工具
  skills?: Skill[];         // 自定义技能
  agents?: AgentDefinition[]; // 自定义 Agent
  commands?: Command[];     // 自定义 /slash 命令
  hooks?: HookDefinition[]; // 自定义 hooks
};
```

### 插件安全

```typescript
// 插件来源分类
type PluginSource = 
  | 'builtin'          // 内置（完全信任）
  | 'plugin'           // 已安装的插件（管理员信任）
  | 'policySettings'   // 企业策略（管理员信任）
  | 'user'             // 用户自定义（受限信任）

// strictPluginOnlyCustomization 模式下：
// - 只有 builtin/plugin/policySettings 来源的扩展可以加载
// - 用户自定义的 MCP、hooks、agents 被阻止
```

## 6.7 ToolSearch — 延迟加载工具

当工具数量很多时（内置 + MCP），全部发送给 API 会消耗大量 token。
ToolSearch 机制允许延迟加载：

```
初始请求：
  tools: [
    Bash (完整 schema),
    Read (完整 schema),
    Edit (完整 schema),
    ToolSearch (完整 schema),  ← 搜索工具
    mcp__db__query (defer_loading: true),  ← 只有名称
    mcp__api__fetch (defer_loading: true),  ← 只有名称
    ... 更多延迟加载的工具
  ]

当 Claude 需要使用延迟加载的工具时：
  1. Claude 调用 ToolSearch({ query: 'database query' })
  2. ToolSearch 返回匹配的工具列表
  3. 下次请求中包含完整的工具 schema
  4. Claude 可以正常调用该工具
```

### alwaysLoad 标记

```typescript
// 某些 MCP 工具可以标记为 alwaysLoad
// 通过 MCP 工具的 _meta['anthropic/alwaysLoad'] = true
// 这些工具始终包含完整 schema，不需要 ToolSearch
```

## ✅ 本章小结

| 概念 | 关键点 |
|------|--------|
| MCP 协议 | 标准化的 AI-工具交互协议，支持 Tools/Resources/Prompts |
| 传输方式 | Stdio, SSE, HTTP, WebSocket |
| 工具包装 | MCP 工具被包装为 `mcp__server__tool` 格式的内置工具 |
| 配置层级 | 用户级 → 项目级 → 企业策略 → Claude.ai |
| Skills | 可复用的 prompt 模板，Markdown 格式 |
| Plugins | 完整的扩展包，可提供工具/技能/Agent/命令/hooks |
| ToolSearch | 延迟加载机制，减少初始 token 消耗 |

### 恭喜！🎉
你已经完成了 Claude Code 核心架构的学习！

### 推荐的进阶学习路径
1. **BashTool 安全系统** — `tools/BashTool/bashSecurity.ts` (100KB)
2. **React/Ink UI** — `components/` 目录
3. **Bridge/IDE 集成** — `bridge/` 目录
4. **记忆系统** — `memdir/` + `services/extractMemories/`
5. **会话恢复** — `utils/conversationRecovery.ts` + `utils/sessionStorage.ts`